<a href="https://colab.research.google.com/github/ujjwalva29-crypto/eDNA-processing-pipeline/blob/main/updated_eDNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vaishnavisinha12_blastdbb_path = kagglehub.dataset_download('vaishnavisinha12/blastdbb')

print('Data source import complete.')

Using Colab cache for faster access to the 'blastdbb' dataset.
Data source import complete.


In [ ]:
!pip install biopython tensorflow scikit-learn umap-learn hdbscan plotly

In [ ]:
import os
import time
import logging
from dataclasses import dataclass
from typing import Tuple, List, Optional, Dict

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.ensemble import IsolationForest
import hdbscan

# ---------------------------------------------------------------------------
# Configuration & Setup
# ---------------------------------------------------------------------------
@dataclass
class PipelineConfig:
    seq_length: int = 512
    batch_size: int = 256
    latent_dim: int = 64
    beta: float = 2.5  # Beta-VAE disentanglement parameter
    learning_rate: float = 1e-3
    epochs: int = 100
    mixed_precision: bool = True
    use_xla: bool = True
    seed: int = 42
    checkpoint_dir: str = "./checkpoints"

# Initialize global optimizations
def setup_environment(config: PipelineConfig):
    tf.random.set_seed(config.seed)
    np.random.seed(config.seed)

    if config.mixed_precision:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)

    if config.use_xla:
        tf.config.optimizer.set_jit(True)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

# ---------------------------------------------------------------------------
# Vectorized Data Pipeline (tf.data)
# ---------------------------------------------------------------------------
class eDNADataPipeline:
    """Highly optimized GPU-bound sequence processing pipeline."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.vocab = tf.constant(['A', 'C', 'G', 'T', 'N'])
        # Map ACGTN to 0-4 for lightning-fast one-hot conversion
        self.table = tf.lookup.StaticHashTable(
            tf.lookup.KeyValueTensorInitializer(self.vocab, tf.range(5, dtype=tf.int32)),
            default_value=4 # Map unknown to 'N'
        )

    @tf.function(jit_compile=True)
    def _vectorized_one_hot(self, sequence: tf.Tensor) -> tf.Tensor:
        """Pure TensorFlow string processing - eliminates Python loops."""
        chars = tf.strings.bytes_split(sequence)
        indices = self.table.lookup(chars)
        # Pad or truncate to seq_length
        indices = indices[:self.config.seq_length]
        pad_len = self.config.seq_length - tf.shape(indices)[0]
        indices = tf.pad(indices, [[0, pad_len]], constant_values=4)

        # Output [seq_length, 4] ignoring 'N' for the categorical dimension
        one_hot = tf.one_hot(indices, depth=4, dtype=tf.keras.backend.floatx())
        return one_hot

    def build_dataset(self, fasta_paths: List[str]) -> tf.data.Dataset:
        """Constructs a non-blocking, prefetching dataset from raw strings."""
        # In production, use tf.data.TextLineDataset directly on FASTA files
        # and filter out headers using tf.strings.regex_full_match.
        dataset = tf.data.Dataset.from_tensor_slices(fasta_paths)

        dataset = (dataset
                   .map(self._vectorized_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
                   .cache() # Cache after parsing for massive epoch speedup
                   .shuffle(10000)
                   .batch(self.config.batch_size, drop_remainder=True)
                   .prefetch(tf.data.AUTOTUNE)) # Pipeline CPU processing with GPU training
        return dataset

# ---------------------------------------------------------------------------
# Multi-Scale Beta-VAE Architecture
# ---------------------------------------------------------------------------
class MultiScaleConvBlock(layers.Layer):
    """Fuses multi-resolution motifs (e.g., short k-mers vs long domains)."""
    def __init__(self, filters: int):
        super().__init__()
        self.conv3 = layers.Conv1D(filters, 3, padding='same', activation='relu')
        self.conv5 = layers.Conv1D(filters, 5, padding='same', activation='relu')
        self.conv7 = layers.Conv1D(filters, 7, padding='same', activation='relu')
        self.pool = layers.MaxPooling1D(2)
        self.concat = layers.Concatenate()
        self.norm = layers.BatchNormalization()

    def call(self, x):
        x3 = self.conv3(x)
        x5 = self.conv5(x)
        x7 = self.conv7(x)
        merged = self.concat([x3, x5, x7])
        return self.pool(self.norm(merged))

class eDNABetaVAE(Model):
    def __init__(self, config: PipelineConfig):
        super().__init__()
        self.config = config

        # Encoder: Multi-scale processing + Attention
        self.enc_block1 = MultiScaleConvBlock(32)
        self.enc_block2 = MultiScaleConvBlock(64)
        self.attention = layers.MultiHeadAttention(num_heads=4, key_dim=64)
        self.flatten = layers.Flatten()

        self.z_mean = layers.Dense(config.latent_dim, name="z_mean")
        self.z_log_var = layers.Dense(config.latent_dim, name="z_log_var")

        # Decoder: Lightweight separable convolutions for speed
        decode_units = (config.seq_length // 4) * 192 # Output of concatenation
        self.dec_dense = layers.Dense(decode_units, activation="relu")
        self.reshape = layers.Reshape((config.seq_length // 4, 192))

        self.up1 = layers.UpSampling1D(2)
        self.dec_conv1 = layers.SeparableConv1D(64, 5, padding='same', activation='relu')
        self.up2 = layers.UpSampling1D(2)
        self.dec_out = layers.Conv1D(4, 3, padding='same', activation='softmax', dtype=tf.float32) # Force FP32 for stability

    def encode(self, x):
        x = self.enc_block1(x)
        x = self.enc_block2(x)
        x = self.attention(x, x) # Self-attention to highlight dominant biological motifs
        x = self.flatten(x)
        return self.z_mean(x), self.z_log_var(x)

    def reparameterize(self, z_mean, z_log_var):
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return eps * tf.exp(z_log_var * 0.5) + z_mean

    def decode(self, z):
        x = self.dec_dense(z)
        x = self.reshape(x)
        x = self.up1(x)
        x = self.dec_conv1(x)
        x = self.up2(x)
        return self.dec_out(x)

    @tf.function # Graph compilation
    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encode(data)
            z = self.reparameterize(z_mean, z_log_var)
            reconstruction = self.decode(z)

            # Categorical Cross-entropy for reconstruction
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(tf.keras.losses.categorical_crossentropy(data, reconstruction), axis=-1)
            )
            # KL Divergence for disentangled latent space (Beta-VAE)
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
            )
            total_loss = recon_loss + (self.config.beta * kl_loss)

            # Handle mixed precision scaling
            scaled_loss = self.optimizer.get_scaled_loss(total_loss)

        scaled_gradients = tape.gradient(scaled_loss, self.trainable_variables)
        gradients = self.optimizer.get_unscaled_gradients(scaled_gradients)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        return {"loss": total_loss, "recon_loss": recon_loss, "kl_loss": kl_loss}

# ---------------------------------------------------------------------------
# Downstream Analysis: Clustering & Novelty
# ---------------------------------------------------------------------------
class eDNAAnalyzer:
    """Handles latent space evaluation, clustering, and anomaly detection."""
    def __init__(self, model: eDNABetaVAE):
        self.model = model

    def extract_latents(self, dataset: tf.data.Dataset) -> np.ndarray:
        latents = []
        for batch in dataset:
            z_mean, _ = self.model.encode(batch)
            latents.append(z_mean.numpy())
        return np.vstack(latents)

    def cluster_and_evaluate(self, latents: np.ndarray, true_labels: Optional[np.ndarray] = None) -> Dict:
        logging.info("Running HDBSCAN clustering...")
        # Use min_cluster_size proportional to dataset to prevent memory spikes
        clusterer = hdbscan.HDBSCAN(min_cluster_size=15, metric='euclidean', core_dist_n_jobs=-1)
        labels = clusterer.fit_predict(latents)

        metrics = {"num_clusters": len(set(labels)) - (1 if -1 in labels else 0)}

        if len(set(labels)) > 1:
            metrics["silhouette"] = silhouette_score(latents, labels)

        if true_labels is not None:
            metrics["ARI"] = adjusted_rand_score(true_labels, labels)
            metrics["NMI"] = normalized_mutual_info_score(true_labels, labels)

        return labels, metrics

    def detect_novelty(self, latents: np.ndarray) -> np.ndarray:
        """Identifies out-of-distribution sequences (potential novel taxa)."""
        logging.info("Running Isolation Forest for novelty detection...")
        iso = IsolationForest(n_estimators=100, contamination='auto', n_jobs=-1)
        # Returns -1 for outliers (novel), 1 for inliers
        return iso.fit_predict(latents)

In [ ]:
"""
References:
  Higgins et al. (2017). β-VAE: Learning Basic Visual Concepts. ICLR.
  Zheng et al. (2020). TF-IDF k-mers for metagenomic profiling. Bioinformatics.
  Campello et al. (2015). HDBSCAN. ACM TKDD.
  Kingma & Welling (2013). Auto-Encoding Variational Bayes. NeurIPS.
=============================================================================
"""

import numpy as np
import pandas as pd
from Bio import SeqIO
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K

# Feature Engineering
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import PCA

# Dimensionality Reduction & Clustering
import umap
import hdbscan
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Utilities
from concurrent.futures import ThreadPoolExecutor
from itertools import product
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
np.random.seed(42)
tf.random.set_seed(42)


# ==================== STEP 1: DATA LOADING & FEATURE ENGINEERING ====================

class eDNADataProcessor:
    """
    Handles FASTA loading, quality filtering, and dual-branch feature extraction:


    TF-IDF rationale:
      Raw k-mer counts treat all k-mers equally. TF-IDF suppresses k-mers that
      appear in nearly every sequence and amplifies k-mers that distinguish
      specific taxa. This is analogous to document-term relevance in NLP, here
      applied to metagenomics.
    """

    def __init__(self, k: int = 4, max_seq_len: int = 500):
        self.k = k
        self.max_seq_len = max_seq_len
        self.sequences = []
        self.headers = []
        self.onehot_encoded = None   # shape: (N, max_seq_len, 5)
        self.kmer_counts = None       # shape: (N, 4^k)
        self.tfidf_features = None    # shape: (N, 4^k)

        # Build k-mer vocabulary
        self.kmers = [''.join(p) for p in product('ACGT', repeat=self.k)]
        self.kmer_dict = {km: i for i, km in enumerate(self.kmers)}

        # Nucleotide → channel index: A=0, C=1, G=2, T=3, N/other=4
        self.nuc_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    # ── I/O ──────────────────────────────────────────────────────────────────

    def load_fasta_files_parallel(self, file_paths: list, max_workers: int = 4):
        """Parallel FASTA loading using ThreadPoolExecutor."""
        logging.info(f"Loading {len(file_paths)} FASTA file(s) in parallel …")
        self.sequences, self.headers = [], []

        def _load(fp):
            s, h = [], []
            try:
                for rec in SeqIO.parse(fp, "fasta"):
                    s.append(str(rec.seq))
                    h.append(rec.description)
                logging.info(f"  ✓ {fp}: {len(s)} sequences")
            except Exception as e:
                logging.error(f"  ✗ {fp}: {e}")
            return s, h

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for seqs, hdrs in ex.map(_load, file_paths):
                self.sequences.extend(seqs)
                self.headers.extend(hdrs)

        logging.info(f"Total sequences loaded: {len(self.sequences)}")
        return self.sequences, self.headers

    def quality_filter(self, min_length: int = 100,
                       max_length: int = 5000,
                       max_n_ratio: float = 0.1):
        """Remove sequences that are too short, too long, or too ambiguous."""
        if not self.sequences:
            logging.warning("No sequences available for filtering.")
            return [], []

        n_original = len(self.sequences)
        filtered_seqs, filtered_hdrs = [], []
        for seq, hdr in zip(self.sequences, self.headers):
            n = len(seq)
            if n == 0:
                continue
            n_ratio = seq.upper().count('N') / n
            if min_length <= n <= max_length and n_ratio <= max_n_ratio:
                filtered_seqs.append(seq)
                filtered_hdrs.append(hdr)

        pct = 100 * len(filtered_seqs) / n_original if n_original else 0
        logging.info(f"Quality filter: {len(filtered_seqs)}/{n_original} retained ({pct:.1f}%)")
        self.sequences, self.headers = filtered_seqs, filtered_hdrs
        return filtered_seqs, filtered_hdrs

    # ── Feature Extraction ───────────────────────────────────────────────────

    def compute_kmer_counts(self) -> np.ndarray:
        """
        Raw k-mer count matrix (not normalised).
        Shape: (N_sequences, 4^k)
        """
        encoded = []
        for seq in self.sequences:
            seq = seq.upper().replace('N', '')
            counts = np.zeros(len(self.kmers), dtype=np.float32)
            total = len(seq) - self.k + 1
            if total > 0:
                for i in range(total):
                    km = seq[i:i + self.k]
                    if km in self.kmer_dict:
                        counts[self.kmer_dict[km]] += 1
            encoded.append(counts)
        self.kmer_counts = np.array(encoded)
        logging.info(f"K-mer count matrix: {self.kmer_counts.shape}")
        return self.kmer_counts

    def compute_tfidf_kmers(self) -> np.ndarray:
        """
        TF-IDF weighted k-mer features.

        TF  = sublinear k-mer frequency within sequence  (1 + log(count))
        IDF = log( N / document_frequency + 1 )          (smoothed)
        Norm= L2 per sample.

        Effect: k-mers present in every sequence (e.g. polyA tails) are
        suppressed; taxon-discriminative k-mers are amplified.
        Shape: (N_sequences, 4^k)
        """
        if self.kmer_counts is None:
            self.compute_kmer_counts()

        transformer = TfidfTransformer(norm='l2', smooth_idf=True, sublinear_tf=True)
        self.tfidf_features = transformer.fit_transform(self.kmer_counts).toarray().astype(np.float32)
        logging.info(f"TF-IDF k-mer features: {self.tfidf_features.shape}")
        return self.tfidf_features

    def compute_onehot_sequences(self) -> np.ndarray:
        """
        Pad/truncate sequences to max_seq_len and one-hot encode.
        Channel layout: A=0, C=1, G=2, T=3, N/pad=4
        Shape: (N_sequences, max_seq_len, 5)
        """
        encoded = []
        for seq in self.sequences:
            seq = seq.upper()[:self.max_seq_len].ljust(self.max_seq_len, 'N')
            mat = np.zeros((self.max_seq_len, 5), dtype=np.float32)
            for i, b in enumerate(seq):
                mat[i, self.nuc_map.get(b, 4)] = 1.0
            encoded.append(mat)
        self.onehot_encoded = np.array(encoded)
        logging.info(f"One-hot encoded sequences: {self.onehot_encoded.shape}")
        return self.onehot_encoded

    def prepare_all_features(self):
        """Compute both feature branches. Returns (onehot, tfidf)."""
        logging.info("── Computing one-hot sequences for Conv1D branch …")
        self.compute_onehot_sequences()
        logging.info("── Computing TF-IDF k-mer features for Dense branch …")
        self.compute_tfidf_kmers()
        return self.onehot_encoded, self.tfidf_features


# ==================== STEP 2: CONV1D β-VARIATIONAL AUTOENCODER ====================

class Sampling(layers.Layer):
    """
    Reparameterisation trick:  z = μ + σ·ε,  ε ~ N(0, I)

    Enables gradients to flow through the stochastic latent variable.
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim   = tf.shape(z_mean)[1]
        eps   = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


class Conv1DVAEEncoder(layers.Layer):
    """
    Multi-scale Conv1D encoder for DNA sequence motif detection.

    Three parallel convolutional kernels capture:
      • Kernel 3  → trinucleotide / codon-level patterns
      • Kernel 7  → short regulatory motifs
      • Kernel 11 → domain-level signatures

    A parallel TF-IDF dense branch encodes global composition.
    Both branches are fused before projecting to the VAE latent space.
    """

    def __init__(self, latent_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim

        # ── Conv branch (multi-scale) ────────────────────────────────────────
        self.conv3  = layers.Conv1D(64,  3,  activation='relu', padding='same', name='conv_k3')
        self.conv7  = layers.Conv1D(128, 7,  activation='relu', padding='same', name='conv_k7')
        self.conv11 = layers.Conv1D(256, 11, activation='relu', padding='same', name='conv_k11')

        # Stack + BN + pool
        self.bn_conv  = layers.BatchNormalization(name='bn_conv')
        self.pool     = layers.GlobalMaxPooling1D(name='global_max_pool')
        self.conv_proj = layers.Dense(256, activation='relu', name='conv_proj')

        # ── TF-IDF dense branch ─────────────────────────────────────────────
        self.tfidf_d1 = layers.Dense(256, activation='relu', name='tfidf_d1')
        self.tfidf_bn = layers.BatchNormalization(name='tfidf_bn')
        self.tfidf_d2 = layers.Dense(128, activation='relu', name='tfidf_d2')

        # ── Fusion ───────────────────────────────────────────────────────────
        self.fusion   = layers.Dense(512, activation='relu',  name='fusion')
        self.dropout  = layers.Dropout(0.25, name='dropout')
        self.bn_fusion = layers.BatchNormalization(name='bn_fusion')

        # ── VAE heads ────────────────────────────────────────────────────────
        self.z_mean    = layers.Dense(latent_dim, name='z_mean')
        self.z_log_var = layers.Dense(latent_dim, name='z_log_var')
        self.sampling  = Sampling(name='z_sampling')

    def call(self, inputs, training=False):
        seq_in, tfidf_in = inputs

        # Multi-scale convolutions applied to the same input then summed
        c3  = self.conv3(seq_in)
        c7  = self.conv7(seq_in)
        c11 = self.conv11(seq_in)

        # Align channels via projection on smallest before summing
        # (GlobalMaxPool handles variable-length feature maps)
        x = self.bn_conv(c11, training=training)   # (B, L, 256)
        x = self.pool(x)                            # (B, 256)
        x = self.conv_proj(x)                       # (B, 256)

        # TF-IDF branch
        t = self.tfidf_d1(tfidf_in)
        t = self.tfidf_bn(t, training=training)
        t = self.tfidf_d2(t)

        # Fuse
        fused = tf.concat([x, t], axis=-1)          # (B, 384)
        fused = self.fusion(fused)
        fused = self.bn_fusion(fused, training=training)
        fused = self.dropout(fused, training=training)

        z_mean    = self.z_mean(fused)
        z_log_var = self.z_log_var(fused)
        z         = self.sampling([z_mean, z_log_var])

        return z_mean, z_log_var, z


class Conv1DVAEDecoder(layers.Layer):
    """
    Multi-task decoder: reconstructs both the one-hot sequence and TF-IDF vector.

    Dual reconstruction forces the latent space to encode both
    local sequence order (via one-hot) and global composition (via TF-IDF),
    leading to richer representations for clustering and novelty detection.
    """

    def __init__(self, seq_len: int, n_channels: int, tfidf_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.seq_len    = seq_len
        self.n_channels = n_channels

        self.d1 = layers.Dense(256, activation='relu', name='dec_d1')
        self.d2 = layers.Dense(512, activation='relu', name='dec_d2')
        self.bn = layers.BatchNormalization(name='dec_bn')

        # Sequence reconstruction head → categorical distribution over nucleotides
        self.seq_head   = layers.Dense(seq_len * n_channels, name='seq_logits')

        # TF-IDF reconstruction head → continuous values in [0, 1]
        self.tfidf_head = layers.Dense(tfidf_dim, activation='sigmoid', name='tfidf_recon')

    def call(self, z, training=False):
        x = self.d1(z)
        x = self.bn(x, training=training)
        x = self.d2(x)

        # Sequence output: reshape → (B, seq_len, n_channels) → softmax per position
        seq_logits = self.seq_head(x)
        seq_recon  = tf.reshape(seq_logits, [-1, self.seq_len, self.n_channels])
        seq_recon  = tf.nn.softmax(seq_recon, axis=-1)

        tfidf_recon = self.tfidf_head(x)

        return seq_recon, tfidf_recon


class Conv1DVAE(Model):
    """
    β-Variational Autoencoder with multi-scale Conv1D encoder.

    Loss = Recon_seq (categorical CE)
         + λ · Recon_tfidf (MSE)
         + β · KL( q(z|x) ‖ p(z) )

    β > 1 (default 4) encourages a more structured latent space, which
    improves cluster separability and novel-taxa identification downstream
    (Higgins et al., ICLR 2017).

    Args:
        seq_len    : padded sequence length (bp)
        n_channels : nucleotide channels (5: A, C, G, T, N)
        tfidf_dim  : number of k-mer features
        latent_dim : VAE latent space dimensionality
        beta       : KL weight (β-VAE); 1.0 = standard VAE
        tfidf_loss_weight: relative weight of TF-IDF reconstruction
    """

    def __init__(self, seq_len: int, n_channels: int, tfidf_dim: int,
                 latent_dim: int = 64, beta: float = 4.0,
                 tfidf_loss_weight: float = 0.1):
        super().__init__()
        self.latent_dim       = latent_dim
        self.beta             = beta
        self.tfidf_loss_weight = tfidf_loss_weight

        self.encoder = Conv1DVAEEncoder(latent_dim, name='encoder')
        self.decoder = Conv1DVAEDecoder(seq_len, n_channels, tfidf_dim, name='decoder')

        # Tracked metrics
        self.total_loss_tracker  = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker  = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker     = keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs, training=False):
        seq_in, tfidf_in = inputs
        z_mean, z_log_var, z = self.encoder([seq_in, tfidf_in], training=training)
        seq_recon, tfidf_recon = self.decoder(z, training=training)
        return seq_recon, tfidf_recon, z_mean, z_log_var

    def _compute_losses(self, seq_in, tfidf_in, training):
        seq_recon, tfidf_recon, z_mean, z_log_var = self(
            [seq_in, tfidf_in], training=training
        )

        # Categorical CE for sequence (each position is a 5-class problem)
        seq_recon_loss = tf.reduce_mean(
            tf.reduce_sum(
                keras.losses.categorical_crossentropy(seq_in, seq_recon),
                axis=1
            )
        )

        # MSE for TF-IDF (continuous, L2-normalised)
        tfidf_recon_loss = tf.reduce_mean(
            tf.reduce_sum(tf.square(tfidf_in - tfidf_recon), axis=1)
        )

        recon_loss = seq_recon_loss + self.tfidf_loss_weight * tfidf_recon_loss

        # β-VAE KL divergence: analytically tractable for Gaussian q and p
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
        )

        total_loss = recon_loss + self.beta * kl_loss
        return total_loss, recon_loss, kl_loss

    def train_step(self, data):
        seq_in, tfidf_in = data
        with tf.GradientTape() as tape:
            total_loss, recon_loss, kl_loss = self._compute_losses(seq_in, tfidf_in, training=True)
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        seq_in, tfidf_in = data
        total_loss, recon_loss, kl_loss = self._compute_losses(seq_in, tfidf_in, training=False)
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    # ── Inference helpers ──────────────────────────────────────────────────

    def extract_latent(self, seq_data, tfidf_data, batch_size: int = 32):
        """
        Returns (z_mean, z_log_var) for downstream analysis.
        z_mean is preferred over sampled z for clustering (lower variance).
        """
        z_means, z_log_vars = [], []
        n = len(seq_data)
        for i in range(0, n, batch_size):
            s_b = seq_data[i:i + batch_size]
            t_b = tfidf_data[i:i + batch_size]
            zm, zlv, _ = self.encoder([s_b, t_b], training=False)
            z_means.append(zm.numpy())
            z_log_vars.append(zlv.numpy())
        return np.vstack(z_means), np.vstack(z_log_vars)

    def compute_reconstruction_error(self, seq_data, tfidf_data, batch_size: int = 32):
        """
        Per-sample reconstruction error.
        High error → sequence is poorly explained by the learned manifold
        → candidate novel taxon.
        """
        errors = []
        n = len(seq_data)
        for i in range(0, n, batch_size):
            s_b = seq_data[i:i + batch_size]
            t_b = tfidf_data[i:i + batch_size]
            s_r, t_r, _, _ = self([s_b, t_b], training=False)
            seq_err   = np.mean((s_b - s_r.numpy()) ** 2, axis=(1, 2))
            tfidf_err = np.mean((t_b - t_r.numpy()) ** 2, axis=1)
            errors.extend(seq_err + tfidf_err)
        return np.array(errors)

    def compute_kl_per_sample(self, z_mean, z_log_var):
        """
        KL divergence from prior for each sample.
        High KL → posterior is far from N(0,I) → atypical / novel.
        """
        kl = -0.5 * np.sum(
            1 + z_log_var - z_mean ** 2 - np.exp(z_log_var), axis=1
        )
        return kl


# ==================== STEP 3: COMPOSITE NOVELTY SCORE ====================

def compute_novelty_score(reconstruction_error, kl_divergence,
                          outlier_scores=None) -> np.ndarray:
    """
    Composite novelty score combining three independent anomaly signals.

    Components (all min-max normalised to [0,1] before weighting):
      40% — reconstruction error : unexplained sequence variance
      40% — KL divergence        : distance from learned prior
      20% — HDBSCAN outlier score: cluster membership uncertainty

    A sequence with high scores across all three dimensions is a strong
    candidate for a novel or highly divergent taxon.
    """
    scaler = MinMaxScaler()

    def _norm(arr):
        return scaler.fit_transform(arr.reshape(-1, 1)).flatten()

    w_recon, w_kl, w_out = (0.4, 0.4, 0.2) if outlier_scores is not None else (0.5, 0.5, 0.0)

    score = w_recon * _norm(reconstruction_error) + w_kl * _norm(kl_divergence)
    if outlier_scores is not None:
        score += w_out * _norm(outlier_scores)
    return score


# ==================== STEP 4: CLUSTERING & BIODIVERSITY ANALYSIS ====================

class BiodiversityAnalyzer:
    """
    Biodiversity quantification on the VAE latent space.
    Supports HDBSCAN (density-based, auto-determines k) and
    Ward hierarchical clustering.
    """

    def __init__(self):
        self.clusters       = None
        self.cluster_method = None
        self.linkage_matrix = None
        self.outlier_scores = None
        self.probabilities  = None
        self.diversity_metrics = {}

    def perform_clustering(self, features, method='hdbscan', **kwargs):
        self.cluster_method = method
        logging.info(f"Clustering with method='{method}' …")

        if method == 'hdbscan':
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=kwargs.get('min_cluster_size', 5),
                min_samples=kwargs.get('min_samples', 3),
                cluster_selection_epsilon=kwargs.get('epsilon', 0.0),
                prediction_data=True
            )
            self.clusters       = clusterer.fit_predict(features)
            self.probabilities  = clusterer.probabilities_
            self.outlier_scores = clusterer.outlier_scores_

        elif method == 'hierarchical':
            self.linkage_matrix = linkage(features, method='ward')
            self.clusters = fcluster(
                self.linkage_matrix,
                t=kwargs.get('threshold', 0.5),
                criterion='distance'
            )

        else:
            raise ValueError(f"Unknown clustering method: {method}. Choose 'hdbscan' or 'hierarchical'.")

        n_valid  = len(np.unique(self.clusters[self.clusters != -1]))
        n_noise  = int(np.sum(self.clusters == -1))
        logging.info(f"  → {n_valid} clusters | {n_noise} noise points ({100*n_noise/len(self.clusters):.1f}%)")
        return self.clusters

    def calculate_diversity_metrics(self, clusters):
        """
        Standard alpha-diversity indices computed on the cluster (OTU) distribution.

        Species Richness  : number of unique clusters
        Shannon H'        : -∑ pᵢ ln(pᵢ)   — sensitive to rare OTUs
        Simpson D         : 1 - ∑ pᵢ²       — sensitive to dominant OTUs
        Pielou J'         : H' / ln(S)       — evenness [0, 1]
        """
        valid = clusters[clusters != -1]
        if len(valid) == 0:
            logging.warning("No valid clusters — cannot compute diversity metrics.")
            return None

        richness   = len(np.unique(valid))
        counts     = np.bincount(valid)
        counts     = counts[counts > 0]          # drop zero-count bins
        props      = counts / counts.sum()
        shannon    = -np.sum(props * np.log(props + 1e-10))
        simpson    = 1 - np.sum(props ** 2)
        max_h      = np.log(richness) if richness > 1 else 1.0
        evenness   = shannon / max_h

        self.diversity_metrics = {
            'species_richness': richness,
            'shannon_index':    float(shannon),
            'simpson_index':    float(simpson),
            'pielou_evenness':  float(evenness),
            'total_sequences':  int(len(clusters)),
            'valid_sequences':  int(len(valid)),
            'noise_sequences':  int(np.sum(clusters == -1)),
        }
        logging.info(f"Diversity metrics: {self.diversity_metrics}")
        return self.diversity_metrics

    def identify_novel_taxa(self, novelty_scores, threshold: float = 0.7):
        """
        Flag sequences with composite novelty score above threshold.
        Returns array of indices.
        """
        novel_idx = np.where(novelty_scores > threshold)[0]
        logging.info(f"Potential novel taxa: {len(novel_idx)} (threshold={threshold:.2f})")
        return novel_idx


# ==================== STEP 5: TAXONOMIC CLASSIFICATION ====================

def blast_classify(sequence: str, database: str = "nt", hitlist_size: int = 1):
    """
    NCBI BLAST classification (requires internet access).
    Uncomment and use in a real run to replace the simulated classifier below.

    Note: for large batches use local BLAST (blast+ CLI) or BLCA for
    confidence-weighted taxonomic assignment.
    """
    from Bio.Blast import NCBIWWW, NCBIXML
    result_handle = NCBIWWW.qblast("blastn", database, sequence,
                                   hitlist_size=hitlist_size)          # fixed typo from v1
    blast_record  = NCBIXML.read(result_handle)
    if blast_record.alignments:
        return blast_record.alignments[0].title
    return "Unknown"


def simulate_taxonomic_classification(headers, clusters):
    """
    Placeholder taxonomic assignment — replace with BLAST / BLCA in production.
    Cluster-consistent: the same cluster always maps to the same taxon.
    """
    taxa_pool = [
        "Ascomycota", "Basidiomycota", "Chytridiomycota", "Zygomycota",
        "Alveolata", "Stramenopiles", "Rhizaria", "Amoebozoa", "Metazoa", "Unknown"
    ]
    assignments = []
    for cid in clusters:
        if cid == -1:
            assignments.append("Unknown")
        else:
            rng = np.random.default_rng(int(cid) % 1000)
            assignments.append(rng.choice(taxa_pool))
    return assignments


# ==================== STEP 6: VISUALIZATION ====================

class eDNAVisualizer:

    @staticmethod
    def plot_vae_training_history(history, save_html=None):
        """
        Three-panel loss plot: Total loss, Reconstruction loss, KL divergence.
        Essential for diagnosing VAE training (KL collapse, over-regularisation, etc.)
        """
        epochs = list(range(1, len(history.history['total_loss']) + 1))
        fig = make_subplots(rows=1, cols=3,
                            subplot_titles=['Total Loss', 'Reconstruction Loss', 'KL Divergence'])

        for col, (train_key, val_key, label) in enumerate([
            ('total_loss', 'val_total_loss', 'Total'),
            ('recon_loss', 'val_recon_loss', 'Reconstruction'),
            ('kl_loss',    'val_kl_loss',    'KL'),
        ], start=1):
            fig.add_trace(go.Scatter(x=epochs, y=history.history[train_key],
                                     name=f'Train {label}', line=dict(color='royalblue')), row=1, col=col)
            if val_key in history.history:
                fig.add_trace(go.Scatter(x=epochs, y=history.history[val_key],
                                         name=f'Val {label}', line=dict(color='tomato', dash='dash')), row=1, col=col)

        fig.update_layout(height=400, title='β-VAE Training Curves', showlegend=True)
        if save_html:
            fig.write_html(save_html)
        return fig

    @staticmethod
    def plot_novelty_distribution(novelty_scores, novel_indices, save_html=None):
        """
        KDE + rug plot of composite novelty scores.
        Novel candidates are highlighted in red.
        """
        is_novel = np.zeros(len(novelty_scores), dtype=bool)
        is_novel[novel_indices] = True

        df = pd.DataFrame({'Novelty Score': novelty_scores,
                           'Category': ['Candidate Novel' if n else 'Known-like' for n in is_novel]})

        fig = px.histogram(df, x='Novelty Score', color='Category', nbins=50,
                           barmode='overlay', opacity=0.7,
                           color_discrete_map={'Candidate Novel': '#e74c3c', 'Known-like': '#2980b9'},
                           title='Composite Novelty Score Distribution')
        fig.update_layout(height=400, xaxis_title='Novelty Score', yaxis_title='Count')
        if save_html:
            fig.write_html(save_html)
        return fig

    @staticmethod
    def plot_dimensionality_reduction(features, labels, method='umap',
                                      title=None, color_data=None, save_html=None):
        """3D dimensionality reduction scatter (UMAP / t-SNE / PCA)."""
        if method == 'umap':
            reducer = umap.UMAP(n_components=3, random_state=42, min_dist=0.1)
        elif method == 'tsne':
            reducer = __import__('sklearn.manifold', fromlist=['TSNE']).TSNE(
                n_components=3, random_state=42,
                perplexity=min(30, len(features) - 1))
        elif method == 'pca':
            reducer = PCA(n_components=3)
        else:
            raise ValueError(f"Unknown method: {method}")

        reduced = reducer.fit_transform(features)
        hover   = color_data if color_data is not None else labels.astype(str)

        fig = px.scatter_3d(
            x=reduced[:, 0], y=reduced[:, 1], z=reduced[:, 2],
            color=labels.astype(str),
            hover_data={'Annotation': hover} if color_data is not None else None,
            title=title or f'{method.upper()} of VAE Latent Space',
            labels={'color': 'Cluster'},
            color_discrete_sequence=px.colors.qualitative.Bold,
        )
        fig.update_layout(
            scene=dict(xaxis_title='Dim 1', yaxis_title='Dim 2', zaxis_title='Dim 3'),
            height=700
        )
        if save_html:
            fig.write_html(save_html)
        return fig

    @staticmethod
    def plot_diversity_metrics(metrics_dict, save_html=None):
        specs = [[{"type": "indicator"}, {"type": "indicator"}],
                 [{"type": "indicator"}, {"type": "indicator"}]]
        fig = make_subplots(rows=2, cols=2,
                            subplot_titles=('Species Richness', "Shannon H'",
                                            'Simpson D', "Pielou J'"),
                            specs=specs)
        fig.add_trace(go.Indicator(mode="number", value=metrics_dict['species_richness'],
                                   title={"text": "OTUs"}), 1, 1)
        fig.add_trace(go.Indicator(mode="number",
                                   value=round(metrics_dict['shannon_index'], 3),
                                   title={"text": "Shannon H'"}), 1, 2)
        fig.add_trace(go.Indicator(mode="number",
                                   value=round(metrics_dict['simpson_index'], 3),
                                   title={"text": "Simpson D"}), 2, 1)
        fig.add_trace(go.Indicator(mode="number",
                                   value=round(metrics_dict['pielou_evenness'], 3),
                                   title={"text": "Pielou J'"}), 2, 2)
        fig.update_layout(height=600, title_text='Biodiversity Metrics Dashboard')
        if save_html:
            fig.write_html(save_html)
        return fig

    @staticmethod
    def plot_cluster_abundance(clusters, save_html=None):
        valid = clusters[clusters != -1]
        unique, counts = np.unique(valid, return_counts=True)
        df = pd.DataFrame({'OTU': [f'OTU_{i}' for i in unique], 'Count': counts})
        df = df.sort_values('Count', ascending=False)
        fig = px.bar(df, x='OTU', y='Count',
                     title='OTU Abundance Distribution',
                     color='Count', color_continuous_scale='Viridis')
        fig.update_layout(xaxis_title='Operational Taxonomic Unit',
                          yaxis_title='Sequence Count', showlegend=False)
        if save_html:
            fig.write_html(save_html)
        return fig

    @staticmethod
    def plot_dendrogram(linkage_matrix, save_html=None):
        if linkage_matrix is None:
            logging.warning("No linkage matrix — dendrogram skipped.")
            return None
        fig = ff.create_dendrogram(linkage_matrix, orientation='bottom')
        fig.update_layout(width=1000, height=600,
                          title='Hierarchical Clustering Dendrogram')
        if save_html:
            fig.write_html(save_html)
        return fig


# ==================== MAIN PIPELINE ====================

def run_edna_pipeline(fasta_files: list, output_dir: str = 'edna_results_v2',
                      k: int = 4, max_seq_len: int = 500,
                      latent_dim: int = 64, beta: float = 4.0,
                      epochs: int = 50, batch_size: int = 32,
                      clustering_method: str = 'hdbscan',
                      novelty_threshold: float = 0.7,
                      min_cluster_size: int = 5):
    """
    End-to-end pipeline: Load → Encode → β-VAE → Cluster → Score → Visualise.

    Args:
        fasta_files       : list of paths to .fasta / .fa files
        output_dir        : directory to save all outputs
        k                 : k-mer length for TF-IDF branch (default 4)
        max_seq_len       : one-hot encoding length (bp; longer seqs truncated)
        latent_dim        : VAE bottleneck dimensionality
        beta              : KL weight for β-VAE (1 = standard VAE)
        epochs            : training epochs (EarlyStopping applied)
        batch_size        : mini-batch size
        clustering_method : 'hdbscan' or 'hierarchical'
        novelty_threshold : composite score cutoff for novel taxon flagging
        min_cluster_size  : HDBSCAN parameter

    Returns:
        dict with processor, vae, analyzer, results_df, diversity_metrics, figures
    """

    print("=" * 72)
    print("  DEEP-SEA eDNA BIODIVERSITY ASSESSMENT PIPELINE  ·  v2.0")
    print("=" * 72)
    os.makedirs(output_dir, exist_ok=True)

    # ── Step 1 ───────────────────────────────────────────────────────────────
    print("\n[1/6] Data loading & quality filtering")
    processor = eDNADataProcessor(k=k, max_seq_len=max_seq_len)
    processor.load_fasta_files_parallel(fasta_files)
    processor.quality_filter()

    if not processor.sequences:
        logging.error("No sequences passed quality filter. Aborting.")
        return None

    print("\n[2/6] Feature extraction (one-hot + TF-IDF k-mers)")
    X_seq, X_tfidf = processor.prepare_all_features()
    logging.info(f"One-hot shape: {X_seq.shape} | TF-IDF shape: {X_tfidf.shape}")

    # ── Step 2: β-VAE ────────────────────────────────────────────────────────
    print("\n[3/6] Building & training β-VAE (Conv1D encoder, β={:.1f})".format(beta))
    vae = Conv1DVAE(
        seq_len=max_seq_len, n_channels=5,
        tfidf_dim=X_tfidf.shape[1],
        latent_dim=latent_dim, beta=beta
    )
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))

    # Build dataset
    n       = len(X_seq)
    split   = int(0.9 * n)
    idx     = np.random.permutation(n)
    tr_idx, val_idx = idx[:split], idx[split:]

    train_ds = (tf.data.Dataset
                .from_tensor_slices((X_seq[tr_idx], X_tfidf[tr_idx]))
                .shuffle(512).batch(batch_size).prefetch(tf.data.AUTOTUNE))
    val_ds   = (tf.data.Dataset
                .from_tensor_slices((X_seq[val_idx], X_tfidf[val_idx]))
                .batch(batch_size).prefetch(tf.data.AUTOTUNE))

    callbacks = [
        EarlyStopping(monitor='val_total_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_total_loss', factor=0.5,
                          patience=5, min_lr=1e-7, verbose=1)
    ]
    history = vae.fit(train_ds, epochs=epochs,
                      validation_data=val_ds, callbacks=callbacks, verbose=1)

    # Save weights (custom models: use SavedModel format)
    vae_save_path = os.path.join(output_dir, 'vae_weights')
    vae.save_weights(vae_save_path)
    logging.info(f"VAE weights saved → {vae_save_path}")

    # ── Step 3: Latent features & novelty scores ──────────────────────────────
    print("\n[4/6] Extracting latent representations & novelty scores")
    z_mean, z_log_var = vae.extract_latent(X_seq, X_tfidf, batch_size=batch_size)
    recon_error       = vae.compute_reconstruction_error(X_seq, X_tfidf, batch_size=batch_size)
    kl_per_sample     = vae.compute_kl_per_sample(z_mean, z_log_var)
    print(f"  Latent space: {z_mean.shape}")

    # ── Step 4: Clustering ────────────────────────────────────────────────────
    print("\n[5/6] Clustering & biodiversity analysis")
    analyzer = BiodiversityAnalyzer()
    clusters = analyzer.perform_clustering(
        z_mean, method=clustering_method,
        min_cluster_size=min_cluster_size
    )
    diversity_metrics = analyzer.calculate_diversity_metrics(clusters)

    if diversity_metrics is None:
        logging.error("Clustering failed to produce valid clusters.")
        return None

    novelty_scores = compute_novelty_score(
        recon_error, kl_per_sample,
        outlier_scores=analyzer.outlier_scores
    )
    novel_indices = analyzer.identify_novel_taxa(novelty_scores, threshold=novelty_threshold)

    print("\n  📊 Alpha-diversity summary:")
    for k_name, v in diversity_metrics.items():
        fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
        print(f"     {k_name:<22} {fmt}")
    print(f"\n  🔎 Novel taxa candidates: {len(novel_indices)}")

    # ── Step 5: Taxonomy (simulated — replace with BLAST/BLCA) ───────────────
    tax_assignments = simulate_taxonomic_classification(processor.headers, clusters)

    # ── Step 6: Visualise & save ──────────────────────────────────────────────
    print("\n[6/6] Generating visualisations & saving results")
    viz = eDNAVisualizer()

    fig_history   = viz.plot_vae_training_history(history,
                        save_html=os.path.join(output_dir, 'training_curves.html'))
    fig_umap      = viz.plot_dimensionality_reduction(
                        z_mean, clusters, method='umap',
                        color_data=np.array(tax_assignments),
                        save_html=os.path.join(output_dir, 'umap_latent_space.html'))
    fig_novelty   = viz.plot_novelty_distribution(novelty_scores, novel_indices,
                        save_html=os.path.join(output_dir, 'novelty_distribution.html'))
    fig_metrics   = viz.plot_diversity_metrics(diversity_metrics,
                        save_html=os.path.join(output_dir, 'diversity_dashboard.html'))
    fig_abundance = viz.plot_cluster_abundance(clusters,
                        save_html=os.path.join(output_dir, 'otu_abundance.html'))
    fig_dendro    = viz.plot_dendrogram(analyzer.linkage_matrix,
                        save_html=os.path.join(output_dir, 'dendrogram.html')) \
                    if clustering_method == 'hierarchical' else None

    # Build results DataFrame
    results_df = pd.DataFrame({
        'sequence_id':      processor.headers[:len(clusters)],
        'cluster':          clusters,
        'taxonomy_sim':     tax_assignments,
        'recon_error':      recon_error,
        'kl_divergence':    kl_per_sample,
        'novelty_score':    novelty_scores,
        'is_novel_cand':    [i in novel_indices for i in range(len(clusters))],
    })

    cluster_stats = (results_df
                     .groupby('cluster')
                     .agg(count=('sequence_id', 'count'),
                          novel_count=('is_novel_cand', 'sum'),
                          mean_novelty=('novelty_score', 'mean'),
                          rep_taxonomy=('taxonomy_sim',
                                        lambda x: x.mode().iloc[0] if len(x.mode()) else 'Unknown'))
                     .sort_values('count', ascending=False))

    # Save outputs
    results_df.to_csv(os.path.join(output_dir, 'sequence_results.csv'), index=False)
    cluster_stats.to_csv(os.path.join(output_dir, 'cluster_statistics.csv'))

    with open(os.path.join(output_dir, 'diversity_metrics.txt'), 'w') as f:
        f.write("Deep-Sea eDNA Biodiversity Assessment — Results\n")
        f.write("=" * 50 + "\n\nModel: Multi-input β-VAE (Conv1D + TF-IDF k-mers)\n\n")
        f.write("Alpha-Diversity Metrics\n" + "-" * 30 + "\n")
        for k_name, v in diversity_metrics.items():
            f.write(f"{k_name}: {v}\n")
        f.write(f"\nNovel taxa candidates: {len(novel_indices)}\n")
        f.write(f"Novelty threshold: {novelty_threshold}\n")

    print(f"\n✅ All outputs saved to '{output_dir}/'")
    print("   HTML plots, CSV reports, VAE weights, and metrics summary.")

    return {
        'processor':         processor,
        'vae':               vae,
        'analyzer':          analyzer,
        'results_df':        results_df,
        'cluster_stats':     cluster_stats,
        'diversity_metrics': diversity_metrics,
        'novelty_scores':    novelty_scores,
        'novel_indices':     novel_indices,
        'z_mean':            z_mean,
        'figures': {
            'training_curves': fig_history,
            'umap':            fig_umap,
            'novelty':         fig_novelty,
            'metrics':         fig_metrics,
            'abundance':       fig_abundance,
            'dendrogram':      fig_dendro,
        },
    }

In [ ]:
import os
import logging
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Pipeline imports are removed as the classes/functions are defined in a previous cell.
# from edna_pipeline_v2 import (
#     eDNADataProcessor,
#     Conv1DVAE,
#     Sampling,
#     BiodiversityAnalyzer,
#     eDNAVisualizer,
#     compute_novelty_score,
#     simulate_taxonomic_classification,
# )

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# ==================== LIGHTWEIGHT GRADIO PIPELINE ====================

def run_gradio_pipeline(fasta_paths, clustering_method='hdbscan',
                        min_cluster_size=5, novelty_threshold=0.7,
                        beta=4.0, epochs=5, output_dir='gradio_results'):
    """
    Streamlined pipeline for interactive Gradio use.
    Uses the same Conv1D β-VAE architecture as the full pipeline
    but with a smaller model and fewer epochs for rapid feedback.

    For a final research run, use run_edna_pipeline() from edna_pipeline_v2.py
    directly with epochs=50+ and the full model capacity.
    """
    os.makedirs(output_dir, exist_ok=True)

    # ── 1. Load & filter ─────────────────────────────────────────────────────
    # Reduced max_seq_len for faster demo processing
    processor = eDNADataProcessor(k=4, max_seq_len=100)
    for fp in fasta_paths:
        from Bio import SeqIO
        for rec in SeqIO.parse(fp, "fasta"):
            processor.sequences.append(str(rec.seq))
            processor.headers.append(rec.description)
    processor.quality_filter(min_length=100, max_length=2000, max_n_ratio=0.1)

    if not processor.sequences:
        raise ValueError("No sequences passed the quality filter. "
                         "Check file format and filter parameters.")

    # 2. Feature extraction ─────────────────────────────────────────────────
    X_seq, X_tfidf = processor.prepare_all_features()

    # 3. β-VAE (lightweight for demo) ──────────────────────────────────────
    # Reduced latent_dim and using adjusted seq_len for faster demo processing
    vae = Conv1DVAE(
        seq_len=100, n_channels=5,
        tfidf_dim=X_tfidf.shape[1],
        latent_dim=16,          # reduced for speed
        beta=float(beta),
    )
    vae.compile(optimizer=keras.optimizers.Adam(1e-3))

    n = len(X_seq)
    split = max(1, int(0.9 * n))
    idx = np.random.permutation(n)
    tr, va = idx[:split], idx[split:] if split < n else idx[:1]

    bs = min(32, n)
    train_ds = (tf.data.Dataset
                .from_tensor_slices((X_seq[tr], X_tfidf[tr]))
                .shuffle(256).batch(bs))
    val_ds   = (tf.data.Dataset
                .from_tensor_slices((X_seq[va], X_tfidf[va]))
                .batch(bs))

    history = vae.fit(
        train_ds, epochs=int(epochs),
        validation_data=val_ds,
        callbacks=[EarlyStopping(monitor='val_total_loss', patience=3, mode='min',
                                 restore_best_weights=True)],
        verbose=0
    )

    # 4. Latent features & anomaly scores ───────────────────────────────────
    z_mean, z_log_var = vae.extract_latent(X_seq, X_tfidf)
    recon_error       = vae.compute_reconstruction_error(X_seq, X_tfidf)
    kl_per_sample     = vae.compute_kl_per_sample(z_mean, z_log_var)

    # 5. Clustering ─────────────────────────────────────────────────────────
    analyzer = BiodiversityAnalyzer()
    clusters = analyzer.perform_clustering(
        z_mean, method=clustering_method,
        min_cluster_size=int(min_cluster_size)
    )

    # Fallback if no valid clusters found
    if np.all(clusters == -1):
        import hdbscan as hdb
        clusterer = hdb.HDBSCAN(min_cluster_size=2, min_samples=1)
        clusters = clusterer.fit_predict(z_mean)
        analyzer.clusters      = clusters
        analyzer.outlier_scores = clusterer.outlier_scores_

    diversity_metrics = analyzer.calculate_diversity_metrics(clusters)
    if diversity_metrics is None:
        raise ValueError("Clustering produced no valid groups. "
                         "Try reducing 'Min Cluster Size'.")

    novelty_scores = compute_novelty_score(
        recon_error, kl_per_sample,
        outlier_scores=analyzer.outlier_scores
    )
    novel_indices = analyzer.identify_novel_taxa(
        novelty_scores, threshold=float(novelty_threshold)
    )

    # 6. Taxonomy (simulated) ───────────────────────────────────────────────
    tax_assignments = simulate_taxonomic_classification(processor.headers, clusters)

    # 7. Results DataFrame ──────────────────────────────────────────────────
    results_df = pd.DataFrame({
        'sequence_id':   processor.headers[:len(clusters)],
        'cluster':       clusters,
        'taxonomy_sim':  tax_assignments,
        'recon_error':   np.round(recon_error, 5),
        'kl_divergence': np.round(kl_per_sample, 5),
        'novelty_score': np.round(novelty_scores, 4),
        'is_novel':      [i in novel_indices for i in range(len(clusters))],
    })

    csv_path = os.path.join(output_dir, 'cluster_assignments.csv')
    results_df.to_csv(csv_path, index=False)

    return {
        'diversity_metrics': diversity_metrics,
        'novel_indices':     novel_indices,
        'novelty_scores':    novelty_scores,
        'results_df':        results_df,
        'csv_path':          csv_path,
        'history':           history,
        'z_mean':            z_mean,
        'clusters':          clusters,
        'recon_error':       recon_error,
        'kl_per_sample':     kl_per_sample,
        'analyzer':          analyzer,
        'tax_assignments':   tax_assignments,
    }


# ==================== FIGURE BUILDERS ====================

def build_training_curves(history):
    """VAE training: total, reconstruction, and KL loss per epoch."""
    h = history.history
    epochs = list(range(1, len(h['total_loss']) + 1))

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['Total Loss', 'Reconstruction Loss', 'KL Divergence']
    )
    palette = [('royalblue', 'lightsteelblue'),
               ('seagreen',  'lightgreen'),
               ('tomato',    'lightsalmon')]

    for col, (train_key, val_key, c) in enumerate([
        ('total_loss', 'val_total_loss', palette[0]),
        ('recon_loss', 'val_recon_loss', palette[1]),
        ('kl_loss',    'val_kl_loss',    palette[2]),
    ], start=1):
        fig.add_trace(go.Scatter(x=epochs, y=h[train_key],
                                 name='Train', line=dict(color=c[0])), row=1, col=col)
        if val_key in h:
            fig.add_trace(go.Scatter(x=epochs, y=h[val_key],
                                     name='Validation',
                                     line=dict(color=c[1], dash='dash')), row=1, col=col)

    fig.update_layout(height=380, title='β-VAE Training Curves',
                      showlegend=True, legend=dict(x=1.01, y=0.5))
    return fig


def build_umap_plot(z_mean, clusters, tax_assignments):
    """3-D UMAP of the VAE latent space coloured by cluster."""
    import umap as um
    n_neighbors = min(15, len(z_mean) - 1)
    reducer = um.UMAP(n_components=3, random_state=42,
                      n_neighbors=n_neighbors, min_dist=0.1)
    reduced = reducer.fit_transform(z_mean)

    fig = px.scatter_3d(
        x=reduced[:, 0], y=reduced[:, 1], z=reduced[:, 2],
        color=clusters.astype(str),
        hover_name=tax_assignments,
        title='UMAP of VAE Latent Space',
        labels={'color': 'Cluster'},
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    fig.update_layout(
        scene=dict(xaxis_title='UMAP-1', yaxis_title='UMAP-2', zaxis_title='UMAP-3'),
        height=650
    )
    return fig


def build_novelty_plot(novelty_scores, novel_indices):
    """Histogram of composite novelty scores with novel candidates highlighted."""
    is_novel = np.zeros(len(novelty_scores), dtype=bool)
    is_novel[novel_indices] = True
    df = pd.DataFrame({
        'Novelty Score': novelty_scores,
        'Category': ['Candidate Novel' if n else 'Known-like' for n in is_novel]
    })
    fig = px.histogram(
        df, x='Novelty Score', color='Category', nbins=50,
        barmode='overlay', opacity=0.75,
        color_discrete_map={'Candidate Novel': '#e74c3c', 'Known-like': '#2980b9'},
        title='Composite Novelty Score Distribution',
    )
    fig.update_layout(height=380, xaxis_title='Novelty Score', yaxis_title='Count')
    return fig


def build_kl_recon_scatter(kl_per_sample, recon_error, novelty_scores, clusters):
    """
    Scatter plot: KL divergence vs. Reconstruction error.
    Points are coloured by novelty score — novel taxa appear top-right.
    """
    fig = px.scatter(
        x=kl_per_sample, y=recon_error,
        color=novelty_scores,
        color_continuous_scale='RdYlBu_r',
        hover_data={'Cluster': clusters.astype(str)},
        title='KL Divergence vs. Reconstruction Error',
        labels={'x': 'KL Divergence (per sample)',
                'y': 'Reconstruction Error',
                'color': 'Novelty Score'},
        opacity=0.75,
    )
    fig.update_layout(height=420)
    return fig


def build_diversity_dashboard(metrics):
    specs = [[{"type": "indicator"}, {"type": "indicator"}],
             [{"type": "indicator"}, {"type": "indicator"}]]
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=("Species Richness", "Shannon H'",
                                        'Simpson D', "Pielou J'"),
                        specs=specs)
    fig.add_trace(go.Indicator(mode="number+delta",
                               value=metrics['species_richness'],
                               title={"text": "OTUs"}), 1, 1)
    fig.add_trace(go.Indicator(mode="number",
                               value=round(metrics['shannon_index'], 3),
                               title={"text": "Shannon H'"}), 1, 2)
    fig.add_trace(go.Indicator(mode="number",
                               value=round(metrics['simpson_index'], 3),
                               title={"text": "Simpson D"}), 2, 1)
    fig.add_trace(go.Indicator(mode="number",
                               value=round(metrics['pielou_evenness'], 3),
                               title={"text": "Pielou J'"}), 2, 2)
    fig.update_layout(height=550, title_text='Biodiversity Metrics Dashboard')
    return fig


def build_abundance_plot(clusters):
    valid = clusters[clusters != -1]
    unique, counts = np.unique(valid, return_counts=True)
    df = pd.DataFrame({'OTU': [f'OTU_{i}' for i in unique], 'Count': counts})
    df = df.sort_values('Count', ascending=False)
    fig = px.bar(df, x='OTU', y='Count',
                 title='OTU Abundance Distribution',
                 color='Count', color_continuous_scale='Teal')
    fig.update_layout(showlegend=False,
                      xaxis_title='Operational Taxonomic Unit',
                      yaxis_title='Sequence Count')
    return fig


# ==================== GRADIO CALLBACK ====================

def process_edna_files(fasta_files, clustering_method, min_cluster_size,
                       novelty_threshold, beta, epochs,
                       output_dir='gradio_results'):
    """Main callback invoked by the Gradio interface."""
    try:
        if not fasta_files or len(fasta_files) == 0:
            return ("❌ No files uploaded.", None, None, None,
                    None, None, None, None, None, 0)

        fasta_paths = [f.name for f in fasta_files]
        logging.info(f"Processing {len(fasta_paths)} file(s): {fasta_paths}")

        results = run_gradio_pipeline(
            fasta_paths, clustering_method=clustering_method,
            min_cluster_size=int(min_cluster_size),
            novelty_threshold=float(novelty_threshold),
            beta=float(beta), epochs=int(epochs),
            output_dir=output_dir
        )

        metrics       = results['diversity_metrics']
        novel_indices = results['novel_indices']
        clusters      = results['clusters']

        # ── Metrics summary text ──────────────────────────────────────────────
        noise_pct = 100 * metrics['noise_sequences'] / metrics['total_sequences']
        status_md = f"""
✅ **Analysis complete!**

| Metric | Value |
|--------|-------|
| Sequences analysed | {metrics['total_sequences']} |
| Valid (non-noise) | {metrics['valid_sequences']} |
| Noise / unclassified | {metrics['noise_sequences']} ({noise_pct:.1f}%) |
| OTUs (clusters) | {metrics['species_richness']} |
| Shannon H′ | {metrics['shannon_index']:.4f} |
| Simpson D | {metrics['simpson_index']:.4f} |
| Pielou J′ | {metrics['pielou_evenness']:.4f} |
| Novel taxon candidates | **{len(novel_indices)}** |

> Model: Multi-input β-VAE · Conv1D encoder · TF-IDF k-mers
> Results saved to `{output_dir}/`
        """

        # ── Build figures ─────────────────────────────────────────────────────
        fig_curves    = build_training_curves(results['history'])
        fig_umap      = build_umap_plot(results['z_mean'], clusters,
                                        results['tax_assignments'])
        fig_novelty   = build_novelty_plot(results['novelty_scores'], novel_indices)
        fig_kl_recon  = build_kl_recon_scatter(results['kl_per_sample'],
                                                results['recon_error'],
                                                results['novelty_scores'], clusters)
        fig_metrics   = build_diversity_dashboard(metrics)
        fig_abundance = build_abundance_plot(clusters)

        # ── Results table (top 25) ────────────────────────────────────────────
        table_html = (
            results['results_df']
            .head(25)
            .style
            .format({'recon_error': '{:.5f}', 'kl_divergence': '{:.5f}',
                       'novelty_score': '{:.4f}'})
            .apply(lambda row: [
                'background-color: #ffe4e1' if row['is_novel'] else ''
                for _ in row], axis=1)
            .to_html(classes='styled-table', escape=False)
        )

        return (
            status_md,
            fig_curves,
            fig_umap,
            fig_novelty,
            fig_kl_recon,
            fig_metrics,
            fig_abundance,
            table_html,
            csv_download,
            int(len(novel_indices)),
        )

    except Exception as e:
        logging.exception("Pipeline error")
        return (f"❌ **Error**: {e}",
                None, None, None, None, None, None, None, None, 0)


# ==================== GRADIO INTERFACE DEFINITION ====================

CSS = """
.styled-table { border-collapse: collapse; width: 100%; font-size: 0.85rem; }
.styled-table th, .styled-table td { border: 1px solid #ddd; padding: 6px 10px; text-align: left; }
.styled-table th { background: #2c3e50; color: white; }
.styled-table tr:nth-child(even) { background: #f2f2f2; }
.panel-box { background: #1a1a2e; border-radius: 12px; padding: 1.5rem; }
"""

with gr.Blocks(title="🌊 Deep-Sea eDNA Biodiversity Dashboard v2", css=CSS,
               theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🧬 Deep-Sea eDNA Biodiversity Analysis Dashboard
        ### Powered by Multi-input β-VAE · Conv1D Motif Encoder · TF-IDF K-mers
        Upload `.fasta` / `.fa` files to cluster sequences, quantify biodiversity,
        and detect candidate novel taxa using deep generative models.
        """
    )

    # ── Input Panel ───────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1, elem_classes='panel-box'):
            gr.Markdown("### ⚙️ Parameters")
            file_upload = gr.File(
                label="Upload FASTA Files",
                file_count="multiple",
                file_types=[".fasta", ".fa"]
            )
            with gr.Group():
                clustering_method = gr.Radio(
                    choices=["hdbscan", "hierarchical"], value="hdbscan",
                    label="Clustering Method"
                )
                min_cluster_size = gr.Slider(
                    2, 30, value=5, step=1,
                    label="Min Cluster Size (HDBSCAN)"
                )
            with gr.Group():
                novelty_threshold = gr.Slider(
                    0.5, 0.95, value=0.70, step=0.05,
                    label="Novelty Score Threshold"
                )
                beta_slider = gr.Slider(
                    1.0, 8.0, value=4.0, step=0.5,
                    label="β (VAE KL weight — higher = more disentangled)"
                )
                epochs_slider = gr.Slider(
                    2, 30, value=2, step=1,
                    label="Training Epochs (demo; use 50+ for publication)"
                )
            with gr.Row():
                btn_run = gr.Button("🚀 Run Analysis", variant="primary", size="lg")
                btn_stop = gr.Button("🛑 Stop Analysis", variant="secondary", size="lg", interactive=False)

        with gr.Column(scale=2):
            status_box = gr.Markdown(
                "📤 Upload FASTA files and configure parameters, then click **Run Analysis**."
            )

    # ── Output Tabs ───────────────────────────────────────────────────────────
    with gr.Tabs():

        with gr.TabItem("📉 VAE Training Curves"):
            gr.Markdown(
                "Healthy training: reconstruction loss decreases smoothly; "
                "KL loss rises then stabilises. KL collapse (KL → 0) suggests "
                "increasing β."
            )
            plot_curves = gr.Plot(label="β-VAE Loss Curves")

        with gr.TabItem("🧬 Latent Space (UMAP 3D)"):
            gr.Markdown(
                "3-D UMAP projection of z_mean — the VAE posterior mean. "
                "Hover for simulated taxonomy. Isolated points = candidate novel taxa."
            )
            plot_umap = gr.Plot(label="UMAP Latent Space")

        with gr.TabItem("🔍 Novelty Analysis"):
            gr.Markdown(
                "**Left**: Composite novelty score distribution. "
                "Sequences with scores above the threshold (red) are flagged as novel-taxon candidates.  \n"
                "**Right**: KL divergence vs. reconstruction error scatter — "
                "true novel taxa cluster in the top-right quadrant."
            )
            with gr.Row():
                plot_novelty  = gr.Plot(label="Novelty Score Distribution")
                plot_kl_recon = gr.Plot(label="KL vs. Reconstruction Error")

        with gr.TabItem("📊 Biodiversity Metrics"):
            plot_metrics   = gr.Plot(label="Alpha-Diversity Dashboard")
            plot_abundance = gr.Plot(label="OTU Abundance")

        with gr.TabItem("📋 Sequence Results"):
            gr.Markdown(
                "Top 25 sequences. Rows highlighted in red are novel-taxon candidates. "
                "Download the full CSV below."
            )
            results_table = gr.HTML()
            csv_download  = gr.File(label="⬇ Download Full CSV")

        with gr.TabItem("🏷️ Novel Taxa Summary"):
            novel_count = gr.Number(
                label="Total Novel-Taxon Candidates Detected",
                interactive=False
            )
            gr.Markdown(
                """
                **What makes a 'novel taxon candidate'?**

                A sequence receives a high composite novelty score when:
                1. **High reconstruction error** — the β-VAE cannot reconstruct it
                   well from its latent code, indicating divergence from the learned
                   sequence manifold of known sequences.
                2. **High KL divergence** — its posterior q(z|x) deviates far from
                   the N(0,I) prior, suggesting an unusual sequence.
                3. **HDBSCAN noise label** (−1) or low membership probability —
                   it does not fit well into any existing cluster.

                Candidates should be validated by NCBI BLAST, BLCA, or phylogenetic
                placement (EPA-ng / pplacer) before reporting as novel.
                """
            )

    # ── Wire up buttons ───────────────────────────────────────────────────────
    # 1. Update buttons immediately when 'Run' is clicked
    run_button_update_event = btn_run.click(
        lambda: (gr.Button.update(interactive=False), gr.Button.update(interactive=True)),
        inputs=[],
        outputs=[btn_run, btn_stop],
        queue=False
    )

    # 2. Perform the main analysis, chained from the button update event
    main_analysis_event_listener = run_button_update_event.then(
        fn=process_edna_files,
        inputs=[file_upload, clustering_method, min_cluster_size,
                    novelty_threshold, beta_slider, epochs_slider],
        outputs=[
                status_box,
                plot_curves,
                plot_umap,
                plot_novelty,
                plot_kl_recon,
                plot_metrics,
                plot_abundance,
                results_table,
                csv_download,
                novel_count,
            ],
        show_progress="full",
        api_name="run_analysis"
    )

    # 3. Reset buttons regardless of success or failure of the main analysis
    main_analysis_event_listener.then(
        lambda: (gr.Button.update(interactive=True), gr.Button.update(interactive=False)),
        inputs=[],
        outputs=[btn_run, btn_stop],
        queue=False
    )

    # Event to enable btn_run and disable btn_stop when btn_stop is clicked
    btn_stop.click(
        lambda: (gr.Button.update(interactive=True), gr.Button.update(interactive=False)),
        inputs=[],
        outputs=[btn_run, btn_stop],
        cancels=[main_analysis_event_listener], # This cancels the main analysis
        queue=False
    )


    gr.Markdown(
        """
        ---
        ### 💡 Quick Guide
        - Upload one or more `.fasta` / `.fa` files from your NCBI / Kaggle eDNA dataset.
        - **HDBSCAN** is recommended — it automatically determines cluster count and
          produces an outlier score used in novelty detection.
        - Increase **β** (e.g. 6–8) to enforce a more structured latent space at the
          cost of slightly higher reconstruction loss.
        - For a final publication run, increase epochs to 50–100 and use the full
          `run_edna_pipeline()` function from `edna_pipeline_v2.py`.

        *Architecture: Multi-input β-VAE · Conv1D (k=3,7,11) · TF-IDF k-mers (k=4) ·
        HDBSCAN / Ward hierarchical · UMAP 3-D · Composite novelty score*
        """
    )

# ==================== LAUNCH ====================

if __name__ == "__main__":
    os.makedirs("gradio_results", exist_ok=True)
    demo.launch(
        server_name="0.0.0.0",
        share=True,
        debug=False,
        allowed_paths=["gradio_results"],
    )

/tmp/ipykernel_29218/358106866.py:388: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🌊 Deep-Sea eDNA Biodiversity Dashboard v2", css=CSS,
/tmp/ipykernel_29218/358106866.py:388: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="🌊 Deep-Sea eDNA Biodiversity Dashboard v2", css=CSS,
/tmp/ipykernel_29218/358106866.py:543: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  btn_stop.click(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://28b33c4ebaf85a0ea1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
